# 📚 Gerador de PDFs por Objetivo de Estudo

Recorta páginas de qualquer PDF acadêmico conforme um `config.json` e gera um PDF por objetivo, com capa e separadores automáticos.

**Os livros não são fixos** — adicione, remova ou troque livros a qualquer módulo apenas editando o JSON na Célula 4.

> ⚠️ **Número de página:** Use o número **real do arquivo PDF**, não o número impresso no livro. A Célula 7 ajuda a calcular o offset.

## 🔧 Célula 1 — Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado!')

Mounted at /content/drive
✅ Drive montado!


## 📦 Célula 2 — Instalar dependências

In [ ]:
!pip install pypdf reportlab --quiet
print('✅ pypdf e reportlab instalados!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.3 MB/s eta 0:00:00
✅ pypdf e reportlab instalados!


## 📁 Célula 3 — Configurar pasta base

Edite apenas `PASTA_BASE`. A pasta `saida/` é criada automaticamente.

In [ ]:
import os
import json

# ╔══════════════════════════════════╗
# ║   CONFIGURAÇÕES DA TURMA         ║
# ╚══════════════════════════════════╝
FACULDADE = "UNDB"  # ← Altere apenas isto para "CEUMA" ou "UNDB"

# --- Roteador Automático ---
ROUTER = {
    "UNDB": {
        "monitor": "João Gabriel R. Trovão",
        "pasta_base": "/content/drive/MyDrive/Logística - Drive/Tutoria"
    },
    "CEUMA": {
        "monitor": "Gabriel Torquato",
        "pasta_base": "/content/drive/MyDrive/Logística - CEUMA/Tutoria - CEUMA"
    }
}

_cfg = ROUTER.get(FACULDADE)
if not _cfg:
    raise ValueError(f"Faculdade '{FACULDADE}' não reconhecida no ROUTER.")

MONITOR_NOME = _cfg["monitor"]
PASTA_BASE   = _cfg["pasta_base"]

PASTA_LIVROS = PASTA_BASE
PASTA_SAIDA  = os.path.join(PASTA_BASE, 'saida')
CONFIG_PATH  = os.path.join(PASTA_BASE, 'config.json')

os.makedirs(PASTA_SAIDA, exist_ok=True)

print(f'📂 Livros : {PASTA_LIVROS}')
print(f'📂 Saída  : {PASTA_SAIDA}')
print(f'📄 Config : {CONFIG_PATH}')
print()

livros_disponiveis = sorted([f for f in os.listdir(PASTA_LIVROS) if f.endswith('.pdf')])
if livros_disponiveis:
    print(f'📚 {len(livros_disponiveis)} PDF(s) encontrados:')
    for l in livros_disponiveis:
        print(f'   • {l}')
else:
    print('⚠️  Nenhum PDF encontrado.')

📂 Livros : /content/drive/MyDrive/Logística - Drive/Tutoria
📂 Saída  : /content/drive/MyDrive/Logística - Drive/Tutoria/saida
📄 Config : /content/drive/MyDrive/Logística - Drive/Tutoria/config.json

📚 7 PDF(s) encontrados:
   • MS - Doenças relacionadas ao trabalho.pdf
   • MS - Dor relacionada trabalho.pdf
   • MS - Guia de Vigilância em saúde.pdf
   • MS - Protocolo Perda Auditiva.pdf
   • MS - Protocolo_pneumoconioses.pdf
   • MS - Saúde do Trabalhador.pdf
   • MS - protocolo_atencao_saude - Chumbo.pdf


## ✏️ Célula 4 — Definir objetivos e páginas

**Esta é a célula que você edita a cada módulo.**

### Campos por corte:
- `arquivo` → nome exato do PDF
- `capitulo` → ex: `"Cap. 13: Invasão Tumoral e Metástase"`
- `secao` → ex: `"Cascata de invasão e metástase"` (opcional, pode deixar `""`)
- `paginas` → lista de páginas reais do arquivo PDF

In [ ]:
# ✏️ EDITE AQUI
config = [
  {
    "objetivo": "01",
    "titulo": "Epidemiologia dos acidentes por animais peçonhentos no Brasil e sistema de notificação",
    "cortes": [
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. I: Ofidismo",
        "secao": "",
        "nivel": "conceito",
        "paginas": [9, 10, 11]
      },
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. II: Escorpionismo",
        "secao": "",
        "nivel": "conceito",
        "paginas": [37]
      },
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. III: Araneísmo",
        "secao": "",
        "nivel": "conceito",
        "paginas": [45]
      },
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. XIV: Modelo de ficha para notificação de acidente por animais peçonhentos (SINAN)",
        "secao": "",
        "nivel": "clinica",
        "paginas": [107, 108, 109, 110]
      }
    ]
  },
  {
    "objetivo": "02",
    "titulo": "Ofidismo: principais serpentes de importância médica, ação do veneno, quadro clínico, complicações, diagnóstico e prognóstico",
    "ordem_motivo": "Caracterização das serpentes (conceito) → manifestações, complicações e diagnóstico do Guia formam sequência contínua (mecanismo → clínica)",
    "cortes": [
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. I: Ofidismo",
        "secao": "",
        "nivel": "conceito",
        "paginas": [12, 13, 14, 15, 16, 17, 18, 19, 20]
      },
      {
        "arquivo": "MS - Guia de Vigilância em saúde.pdf",
        "capitulo": "Cap. 10: Acidente Ofídico",
        "secao": "",
        "nivel": "clinica",
        "paginas": [1020, 1021, 1022, 1023, 1024]
      }
    ]
  },
  {
    "objetivo": "03",
    "titulo": "Escorpionismo e araneísmo: espécies envolvidas, ação do veneno, quadro clínico e complicações",
    "ordem_motivo": "Manual (espécies e veneno) antes do Guia (clínica); escorpionismo antes do araneísmo mantendo a ordem do roteiro",
    "cortes": [
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. II: Escorpionismo",
        "secao": "",
        "nivel": "conceito",
        "paginas": [38, 39, 40, 41]
      },
      {
        "arquivo": "MS - Guia de Vigilância em saúde.pdf",
        "capitulo": "Cap. 10: Escorpionismo",
        "secao": "",
        "nivel": "clinica",
        "paginas": [1026, 1027, 1028]
      },
      {
        "arquivo": "MS - Manual de Diagnóstico e Tratamento de Acidentes por Animais Peçonhentos.pdf",
        "capitulo": "Cap. III: Araneísmo",
        "secao": "",
        "nivel": "conceito",
        "paginas": [46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]
      },
      {
        "arquivo": "MS - Guia de Vigilância em saúde.pdf",
        "capitulo": "Cap. 10: Araneísmo",
        "secao": "",
        "nivel": "clinica",
        "paginas": [1033, 1034, 1035, 1036]
      }
    ]
  }
]



with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f'✅ config.json salvo com {len(config)} objetivo(s):')
for obj in config:
    n_pags = sum(len(c['paginas']) for c in obj['cortes'])
    print(f'   • Objetivo {obj["objetivo"]}: {n_pags} página(s) de {len(obj["cortes"])} fonte(s)')

✅ config.json salvo com 3 objetivo(s):
   • Objetivo 01: 9 página(s) de 4 fonte(s)
   • Objetivo 02: 14 página(s) de 2 fonte(s)
   • Objetivo 03: 22 página(s) de 4 fonte(s)


## 🔍 Célula 5 — Validar config

Verifica se todos os arquivos existem e se as páginas estão dentro do intervalo válido.

In [ ]:
import os
import json
from pypdf import PdfReader
from difflib import get_close_matches

def _pdfs_na_pasta(pasta):
    """Retorna lista de nomes de PDF existentes na pasta."""
    return [f for f in os.listdir(pasta) if f.lower().endswith('.pdf')]

def _sugerir_nome(arquivo, pdfs_disponiveis):
    """
    Tenta encontrar o PDF mais parecido com o nome informado.
    Retorna o nome sugerido ou None se não encontrar nada próximo.
    """
    matches = get_close_matches(arquivo, pdfs_disponiveis, n=1, cutoff=0.6)
    return matches[0] if matches else None

def validar_config(config, pasta_livros):
    erros = []
    sugestoes = {}  # arquivo errado → sugestão
    pdfs_disponiveis = _pdfs_na_pasta(pasta_livros)

    for obj in config:
        rotulo = f"Objetivo {obj['objetivo']}"
        for corte in obj['cortes']:
            arquivo = corte['arquivo']
            caminho = os.path.join(pasta_livros, arquivo)

            if not os.path.exists(caminho):
                # Tenta autocorreção
                if arquivo not in sugestoes:
                    sugestao = _sugerir_nome(arquivo, pdfs_disponiveis)
                    sugestoes[arquivo] = sugestao

                sugestao = sugestoes[arquivo]
                if sugestao:
                    erros.append(
                        f"[{rotulo}] Arquivo não encontrado: '{arquivo}'\n"
                        f"           💡 Você quis dizer: '{sugestao}'?"
                    )
                else:
                    erros.append(
                        f"[{rotulo}] Arquivo não encontrado: '{arquivo}'\n"
                        f"           ❓ Nenhum PDF parecido encontrado na pasta."
                    )
                continue

            total = len(PdfReader(caminho).pages)
            for pg in corte['paginas']:
                if pg < 1 or pg > total:
                    erros.append(
                        f"[{rotulo}] {arquivo}: página {pg} inválida "
                        f"(arquivo tem {total} páginas)"
                    )
    return erros, sugestoes


with open(CONFIG_PATH, encoding='utf-8') as f:
    config = json.load(f)

erros, sugestoes = validar_config(config, PASTA_LIVROS)

if erros:
    print('❌ Erros encontrados — corrija antes de gerar:\n')
    for e in erros:
        print(f'   • {e}')

    # Resume todas as substituições sugeridas de uma vez
    nomes_errados = {arq: sug for arq, sug in sugestoes.items() if sug}
    if nomes_errados:
        print('\n─────────────────────────────────────────')
        print('🔧 Substituições sugeridas no config.json:')
        for errado, correto in nomes_errados.items():
            print(f'   "{errado}"')
            print(f'   → "{correto}"')
        print('─────────────────────────────────────────')
else:
    print('✅ Tudo validado!')
    print('🔎 Rode a Célula 5.5 para ver o preview.')

✅ Tudo validado!
🔎 Rode a Célula 5.5 para ver o preview.


## 🔎 Célula 5.5 — Preview antes de gerar

Mostra exatamente o que vai entrar em cada PDF, com posição final e capítulos.

In [ ]:
from pypdf import PdfReader
import re

def nome_legivel(arquivo):
    nome = os.path.basename(arquivo).replace('.pdf', '')
    return re.sub(r'[_\-]+', ' ', nome).title()

def formatar_paginas(paginas):
    paginas = sorted(set(paginas))
    grupos, inicio, fim = [], paginas[0], paginas[0]
    for p in paginas[1:]:
        if p == fim + 1: fim = p
        else:
            grupos.append((inicio, fim)); inicio = fim = p
    grupos.append((inicio, fim))
    return ', '.join(str(a) if a == b else f'{a}–{b}' for a, b in grupos)

with open(CONFIG_PATH, encoding='utf-8') as f:
    config = json.load(f)

total_geral = 0
print('=' * 65)
print('📋 PREVIEW — O QUE SERÁ GERADO')
print('=' * 65)

for obj in config:
    n_pags       = sum(len(c['paginas']) for c in obj['cortes'])
    total_pdf    = 1 + len(obj['cortes']) + n_pags
    total_geral += total_pdf

    print(f"\n🎯 Objetivo {obj['objetivo']}  ({total_pdf} págs. no PDF final)")
    titulo_curto = obj['titulo'][:75] + '...' if len(obj['titulo']) > 75 else obj['titulo']
    print(f"   {titulo_curto}")
    print()

    pagina_atual = 2
    for i, corte in enumerate(obj['cortes']):
        nome     = nome_legivel(corte['arquivo'])
        capitulo = corte.get('capitulo', '')
        secao    = corte.get('secao', '')
        n        = len(corte['paginas'])
        pags_str = formatar_paginas(corte['paginas'])
        pg_ini   = pagina_atual + 1  # +1 pelo separador
        pg_fim   = pg_ini + n - 1

        print(f"   [{i+1}] {nome}")
        if capitulo: print(f"       {capitulo}")
        if secao:    print(f"       › {secao}")
        print(f"       Fonte: págs. {pags_str}  →  PDF final: p. {pg_ini}–{pg_fim}")

        pagina_atual += 1 + n

print()
print('=' * 65)
print(f'📦 Total: {total_geral} páginas em {len(config)} PDF(s)')
print('=' * 65)
print('\n✅ Se estiver correto, rode a Célula 6.')

📋 PREVIEW — O QUE SERÁ GERADO

🎯 Objetivo 01  (14 págs. no PDF final)
   Epidemiologia dos acidentes por animais peçonhentos no Brasil e sistema de ...

   [1] Ms   Manual De Diagnóstico E Tratamento De Acidentes Por Animais Peçonhentos
       Cap. I: Ofidismo
       Fonte: págs. 9–11  →  PDF final: p. 3–5
   [2] Ms   Manual De Diagnóstico E Tratamento De Acidentes Por Animais Peçonhentos
       Cap. II: Escorpionismo
       Fonte: págs. 37  →  PDF final: p. 7–7
   [3] Ms   Manual De Diagnóstico E Tratamento De Acidentes Por Animais Peçonhentos
       Cap. III: Araneísmo
       Fonte: págs. 45  →  PDF final: p. 9–9
   [4] Ms   Manual De Diagnóstico E Tratamento De Acidentes Por Animais Peçonhentos
       Cap. XIV: Modelo de ficha para notificação de acidente por animais peçonhentos (SINAN)
       Fonte: págs. 107–110  →  PDF final: p. 11–14

🎯 Objetivo 02  (17 págs. no PDF final)
   Ofidismo: principais serpentes de importância médica, ação do veneno, quadr...

   [1] Ms   Manual De Di

## 🚀 Célula 6 — Gerar os PDFs

Gera um PDF por objetivo com:
- **Capa** com índice de navegação (posição no PDF + capítulo)
- **Separador** entre fontes com nome do livro e capítulo
- **Nome do arquivo**: `Objetivo 01.pdf`, `Objetivo 02.pdf`...

In [ ]:
import io
import re
import os
import json
from pypdf import PdfReader, PdfWriter
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.styles import ParagraphStyle
from reportlab.pdfgen import canvas as rl_canvas

# ── Autoria ────────────────────────────────────────────────────────────────────
AUTORIA = f"© Conteúdo Autoral  •  {MONITOR_NOME}"


def nome_legivel(arquivo):
    nome = os.path.basename(arquivo).replace('.pdf', '')
    return re.sub(r'[_\-]+', ' ', nome).title()


def formatar_paginas(paginas):
    paginas = sorted(set(paginas))
    grupos, inicio, fim = [], paginas[0], paginas[0]
    for p in paginas[1:]:
        if p == fim + 1:
            fim = p
        else:
            grupos.append((inicio, fim))
            inicio = fim = p
    grupos.append((inicio, fim))
    return ', '.join(str(a) if a == b else f'{a}\u2013{b}' for a, b in grupos)


def validar_config(config, pasta_livros):
    erros = []
    for obj in config:
        rotulo = f"Objetivo {obj['objetivo']}"
        paginas_vistas = {}
        for i, corte in enumerate(obj['cortes']):
            arquivo = corte['arquivo']
            caminho = os.path.join(pasta_livros, arquivo)
            if not os.path.exists(caminho):
                erros.append(f"[{rotulo}] Arquivo não encontrado: '{arquivo}'")
                continue
            total = len(PdfReader(caminho).pages)
            # Logo antes do loop "for pg in corte['paginas']:"
            if not isinstance(corte['paginas'], list):
                erros.append(
                    f"[{rotulo}] {arquivo}: campo 'paginas' inválido — "
                    f"esperado lista, encontrado: {repr(corte['paginas'])}"
                )
                continue
            for pg in corte['paginas']:
                if pg < 1:
                    erros.append(
                        f"[{rotulo}] {arquivo}: página {pg} é inválida (≤ 0). "
                        f"Provavelmente offset negativo mal aplicado — verifique o offsets.json."
                    )
                elif pg > total:
                    erros.append(
                        f"[{rotulo}] {arquivo}: página {pg} inválida "
                        f"(arquivo tem {total} páginas)"
                    )
                chave = (arquivo, pg)
                if chave in paginas_vistas:
                    erros.append(
                        f"[{rotulo}] Página {pg} de '{arquivo}' aparece nos cortes "
                        f"{paginas_vistas[chave]+1} e {i+1} — duplicata dentro do mesmo objetivo."
                    )
                else:
                    paginas_vistas[chave] = i
    return erros


def gerar_capa(objetivo, titulo, cortes, pasta_livros, fusao=None):
    """
    Capa com índice de navegação.
    - Fundo branco, detalhes em azul.
    - Se 'fusao' for fornecido, exibe banner informando os objetivos fundidos.
    """
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(
        buffer, pagesize=A4,
        leftMargin=2.5 * cm, rightMargin=2.5 * cm,
        topMargin=3.5 * cm, bottomMargin=2.5 * cm
    )

    azul        = colors.HexColor('#3b5bdb')
    preto       = colors.HexColor('#1a1a2e')
    cinza       = colors.HexColor('#555555')
    cinza_claro = colors.HexColor('#999999')
    divisor     = colors.HexColor('#dddddd')
    azul_claro  = colors.HexColor('#eef1ff')  # fundo do banner de fusão

    s_label  = ParagraphStyle('label',  fontName='Helvetica-Bold', fontSize=9,
                              textColor=azul, spaceAfter=4, leading=12)
    s_num    = ParagraphStyle('num',    fontName='Helvetica-Bold', fontSize=36,
                              textColor=preto, spaceAfter=2, leading=40)
    s_titulo = ParagraphStyle('titulo', fontName='Helvetica-Bold', fontSize=15,
                              textColor=preto, spaceAfter=20, leading=22, wordWrap='LTR')
    s_secao  = ParagraphStyle('secao',  fontName='Helvetica-Bold', fontSize=8,
                              textColor=azul, spaceBefore=16, spaceAfter=10,
                              leading=10, letterSpacing=0.5)
    s_pg     = ParagraphStyle('pg',     fontName='Helvetica-Bold', fontSize=13,
                              textColor=azul, spaceBefore=10, spaceAfter=1, leading=16)
    s_livro  = ParagraphStyle('livro',  fontName='Helvetica-Bold', fontSize=10,
                              textColor=preto, spaceAfter=1, leading=14, wordWrap='LTR')
    s_cap    = ParagraphStyle('cap',    fontName='Helvetica', fontSize=10,
                              textColor=cinza, spaceAfter=1, leading=14,
                              leftIndent=8, wordWrap='LTR')
    s_sec    = ParagraphStyle('sec',    fontName='Helvetica', fontSize=9,
                              textColor=cinza_claro, spaceAfter=4, leading=13,
                              leftIndent=8, wordWrap='LTR')
    s_autoria = ParagraphStyle('autoria', fontName='Helvetica', fontSize=8,
                               textColor=colors.HexColor('#aaaaaa'),
                               alignment=1, leading=12)
    # Estilos do banner de fusão
    s_fusao_titulo = ParagraphStyle('fusao_titulo', fontName='Helvetica-Bold', fontSize=8,
                                    textColor=azul, spaceAfter=4, leading=11,
                                    letterSpacing=0.5)
    s_fusao_texto  = ParagraphStyle('fusao_texto', fontName='Helvetica', fontSize=9,
                                    textColor=preto, spaceAfter=2, leading=13,
                                    leftIndent=8, wordWrap='LTR')

    elems = [
        Paragraph('OBJETIVO DE ESTUDO', s_label),
        Paragraph(objetivo, s_num),
        Paragraph(titulo, s_titulo),
    ]

# ── Banner de fusão (aparece só quando o campo "fusao" existe no JSON) ──────
    if fusao:
        elems.append(HRFlowable(width='100%', thickness=0.5, color=azul, spaceAfter=8))
        elems.append(Paragraph('⚠ ESTE MATERIAL ABRANGE MAIS DE UM OBJETIVO', s_fusao_titulo))
        # fusao deve ser lista de strings com os títulos completos de cada objetivo
        linhas = fusao if isinstance(fusao, list) else [fusao]
        for linha in linhas:
            elems.append(Paragraph(f'• {linha}', s_fusao_texto))
            elems.append(Spacer(1, 0.15 * cm))
        elems.append(Spacer(1, 0.2 * cm))
        elems.append(HRFlowable(width='100%', thickness=0.5, color=azul, spaceAfter=8))
    else:
        elems.append(HRFlowable(width='100%', thickness=0.5, color=divisor, spaceAfter=4))

    elems.append(Paragraph('ÍNDICE', s_secao))

    # Calcula posição real de cada trecho no PDF final
    pagina_atual = 2  # p.1 = capa
    for corte in cortes:
        nome     = nome_legivel(corte['arquivo'])
        capitulo = corte.get('capitulo', '')
        secao    = corte.get('secao', '')
        n        = len(corte['paginas'])
        pg_ini   = pagina_atual + 1   # +1 pelo separador do grupo
        pg_fim   = pg_ini + n - 1

        elems.append(Paragraph(f'p. {pg_ini}\u2013{pg_fim}', s_pg))
        elems.append(Paragraph(nome, s_livro))
        if capitulo:
            elems.append(Paragraph(capitulo, s_cap))
        if secao:
            elems.append(Paragraph(f'\u203a {secao}', s_sec))

        pagina_atual += 1 + n

    elems.append(Spacer(1, 1 * cm))
    elems.append(HRFlowable(width='100%', thickness=0.5, color=divisor, spaceAfter=6))
    elems.append(Paragraph(AUTORIA, s_autoria))

    doc.build(elems)
    buffer.seek(0)
    return buffer


def gerar_separador(nome_livro, capitulo='', secao='', pagesize=A4):
    """
    Página separadora com fundo branco e barra azul lateral.
    - Quebra de linha automática em todos os campos de texto.
    - pagesize pode ser passado para igualar ao tamanho das páginas do livro.
    """
    buffer = io.BytesIO()
    W, H = pagesize
    c = rl_canvas.Canvas(buffer, pagesize=pagesize)

    azul  = colors.HexColor('#3b5bdb')
    preto = colors.HexColor('#1a1a2e')
    cinza = colors.HexColor('#555555')

    # Fundo branco
    c.setFillColor(colors.white)
    c.rect(0, 0, W, H, fill=1, stroke=0)

    # Barra azul vertical
    c.setFillColor(azul)
    c.rect(2 * cm, 0, 0.35 * cm, H, fill=1, stroke=0)

    # Margem esquerda do texto e largura máxima disponível
    x_texto  = 3.2 * cm
    max_larg = W - x_texto - 1.5 * cm  # margem direita de segurança

    def draw_wrapped(c, texto, x, y_topo, fonte, tamanho, cor, line_height):
        """
        Desenha texto com quebra automática de linha.
        Começa em y_topo e desce. Retorna o y final (após última linha).
        """
        c.setFont(fonte, tamanho)
        c.setFillColor(cor)

        palavras = texto.split()
        linha_atual = ''
        linhas = []

        for palavra in palavras:
            teste = (linha_atual + ' ' + palavra).strip()
            if c.stringWidth(teste, fonte, tamanho) <= max_larg:
                linha_atual = teste
            else:
                if linha_atual:
                    linhas.append(linha_atual)
                linha_atual = palavra
        if linha_atual:
            linhas.append(linha_atual)

        y = y_topo
        for linha in linhas:
            c.setFont(fonte, tamanho)
            c.setFillColor(cor)
            c.drawString(x, y, linha)
            y -= line_height

        return y  # y após a última linha

    base = H / 2

    # Label "FONTE"
    c.setFillColor(azul)
    c.setFont('Helvetica-Bold', 8)
    c.drawString(x_texto, base + 2.8 * cm, 'FONTE')

    # Nome do livro (fonte grande, quebra automática)
    y_apos_livro = draw_wrapped(
        c, nome_livro,
        x=x_texto, y_topo=base + 1.8 * cm,
        fonte='Helvetica-Bold', tamanho=18,
        cor=preto, line_height=0.75 * cm
    )

    # Capítulo
    if capitulo:
        y_apos_cap = draw_wrapped(
            c, capitulo,
            x=x_texto, y_topo=y_apos_livro - 0.4 * cm,
            fonte='Helvetica-Bold', tamanho=11,
            cor=preto, line_height=0.45 * cm
        )
    else:
        y_apos_cap = y_apos_livro - 0.2 * cm

    # Seção
    if secao:
        draw_wrapped(
            c, f'\u203a  {secao}',
            x=x_texto, y_topo=y_apos_cap - 0.3 * cm,
            fonte='Helvetica', tamanho=10,
            cor=cinza, line_height=0.4 * cm
        )

    c.save()
    return buffer


def _cortes_precisam_separador(corte_anterior, corte_atual):
    if corte_anterior is None:
        return True
    mesmo_arquivo  = corte_anterior['arquivo']          == corte_atual['arquivo']
    mesmo_capitulo = corte_anterior.get('capitulo', '') == corte_atual.get('capitulo', '')
    paginas_ant    = sorted(corte_anterior['paginas'])
    paginas_atu    = sorted(corte_atual['paginas'])
    contiguas      = paginas_ant and paginas_atu and (paginas_ant[-1] + 1 == paginas_atu[0])
    return not (mesmo_arquivo and mesmo_capitulo and contiguas)


def gerar_pdf_objetivo(obj, pasta_livros, pasta_saida):
    writer = PdfWriter()

    # Lê campo "fusao" do JSON (opcional)
    fusao = obj.get('fusao', None)

    # Capa com índice de navegação
    cortes_capa = _agrupar_cortes_para_capa(obj['cortes'])
    capa = PdfReader(gerar_capa(obj['objetivo'], obj['titulo'], cortes_capa, pasta_livros, fusao=fusao))
    writer.add_page(capa.pages[0])

    # Separadores + páginas
    corte_anterior = None
    for corte in obj['cortes']:
        nome     = nome_legivel(corte['arquivo'])
        capitulo = corte.get('capitulo', '')
        secao    = corte.get('secao', '')

        if _cortes_precisam_separador(corte_anterior, corte):
            sep = PdfReader(gerar_separador(nome, capitulo, secao))
            writer.add_page(sep.pages[0])

        caminho = os.path.join(pasta_livros, corte['arquivo'])
        reader  = PdfReader(caminho)
        total   = len(reader.pages)
        for pg in corte['paginas']:
            idx = pg - 1
            if 0 <= idx < total:
                writer.add_page(reader.pages[idx])
            else:
                print(f'     ⚠️  Página {pg} ignorada (fora do intervalo)')

        corte_anterior = corte

    # Salva
    nome_arquivo  = f"Objetivo {obj['objetivo']}.pdf"
    caminho_saida = os.path.join(pasta_saida, nome_arquivo)
    with open(caminho_saida, 'wb') as f:
        writer.write(f)

    n      = sum(len(c['paginas']) for c in obj['cortes'])
    n_seps = sum(
        1 for i, c in enumerate(obj['cortes'])
        if _cortes_precisam_separador(obj['cortes'][i-1] if i > 0 else None, c)
    )
    total_pdf = 1 + n_seps + n
    fusao_aviso = ' [FUSÃO]' if fusao else ''
    print(f'   ✅ {nome_arquivo}  ({total_pdf} páginas, {n_seps} separador(es)){fusao_aviso}')
    return caminho_saida


def _agrupar_cortes_para_capa(cortes):
    if not cortes:
        return []
    grupos = []
    grupo_atual = {
        'arquivo':  cortes[0]['arquivo'],
        'capitulo': cortes[0].get('capitulo', ''),
        'secao':    cortes[0].get('secao', ''),
        'paginas':  list(cortes[0]['paginas']),
    }
    for corte in cortes[1:]:
        if not _cortes_precisam_separador(grupo_atual, corte):
            grupo_atual['paginas'].extend(corte['paginas'])
            secao_nova = corte.get('secao', '')
            if secao_nova and secao_nova != grupo_atual['secao']:
                grupo_atual['secao'] = (
                    grupo_atual['secao'] + ' / ' + secao_nova
                    if grupo_atual['secao'] else secao_nova
                )
        else:
            grupos.append(grupo_atual)
            grupo_atual = {
                'arquivo':  corte['arquivo'],
                'capitulo': corte.get('capitulo', ''),
                'secao':    corte.get('secao', ''),
                'paginas':  list(corte['paginas']),
            }
    grupos.append(grupo_atual)
    return grupos


# ── EXECUÇÃO ───────────────────────────────────────────────────────────────────
with open(CONFIG_PATH, encoding='utf-8') as f:
    config = json.load(f)

erros = validar_config(config, PASTA_LIVROS)
if erros:
    print('❌ Corrija os erros antes de gerar (rode a Célula 5):')
    for e in erros:
        print(f'   • {e}')
else:
    print(f'🚀 Gerando {len(config)} PDF(s)...\n')
    for obj in config:
        fusao_aviso = ' [FUSÃO]' if obj.get('fusao') else ''
        print(f'📄 Objetivo {obj["objetivo"]}{fusao_aviso} — {obj["titulo"][:60]}...')
        gerar_pdf_objetivo(obj, PASTA_LIVROS, PASTA_SAIDA)
    print(f'\n🎉 Concluído! Arquivos em: {PASTA_SAIDA}')

🚀 Gerando 3 PDF(s)...

📄 Objetivo 01 — Epidemiologia dos acidentes por animais peçonhentos no Brasi...


   ✅ Objetivo 01.pdf  (14 páginas, 4 separador(es))
📄 Objetivo 02 — Ofidismo: principais serpentes de importância médica, ação d...


   ✅ Objetivo 02.pdf  (17 páginas, 2 separador(es))
📄 Objetivo 03 — Escorpionismo e araneísmo: espécies envolvidas, ação do vene...


   ✅ Objetivo 03.pdf  (27 páginas, 4 separador(es))

🎉 Concluído! Arquivos em: /content/drive/MyDrive/Logística - Drive/Tutoria/saida


## 🔎 Célula 7 — Calcular offset de um livro

Use para descobrir a diferença entre página impressa no livro e número real do arquivo PDF.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 7 — Calcular e salvar offsets de todos os livros        ║
# ║  FIX 5: exibe apenas livros presentes na pasta atual            ║
# ╚══════════════════════════════════════════════════════════════════╝

import os
import json
from pypdf import PdfReader

OFFSETS_PATH = os.path.join(PASTA_BASE, 'offsets.json')


def _carregar_offsets():
    if os.path.exists(OFFSETS_PATH):
        with open(OFFSETS_PATH, encoding='utf-8') as f:
            return json.load(f)
    return {}


def _salvar_offsets(offsets):
    with open(OFFSETS_PATH, 'w', encoding='utf-8') as f:
        json.dump(offsets, f, ensure_ascii=False, indent=2)


def _pedir_offset(arquivo, caminho):
    total = len(PdfReader(caminho).pages)
    print(f'\n📖 Mapeando: {arquivo}  ({total} páginas no arquivo)')
    print('   Para calcular o offset, escolha qualquer página com número impresso visível.')

    while True:
        try:
            pg_impressa = int(input('   → Número impresso na página: ').strip())
            pg_leitor   = int(input('   → Número que o leitor de PDF mostra (contador): ').strip())
            break
        except ValueError:
            print('   ⚠️  Digite apenas números inteiros.')

    offset = pg_leitor - pg_impressa
    sinal  = f'+{offset}' if offset >= 0 else str(offset)
    print(f'   ✅ Offset calculado: {sinal}  (impresso {pg_impressa} = arquivo {pg_leitor})')
    return offset, total


def mapear_offsets():
    offsets = _carregar_offsets()
    # Apenas PDFs presentes fisicamente na pasta agora
    livros  = sorted([f for f in os.listdir(PASTA_LIVROS) if f.endswith('.pdf')])

    if not livros:
        print('⚠️  Nenhum PDF encontrado em:', PASTA_LIVROS)
        return

    print('=' * 65)
    print('📐 CÉLULA 7 — OFFSETS DE PÁGINA')
    print('=' * 65)

    algum_novo          = False
    algum_desatualizado = False

    for arquivo in livros:
        caminho     = os.path.join(PASTA_LIVROS, arquivo)
        total_atual = len(PdfReader(caminho).pages)

        if arquivo in offsets:
            entrada = offsets[arquivo]

            if isinstance(entrada, dict):
                offset_salvo = entrada['offset']
                total_salvo  = entrada.get('total_paginas')
            else:
                offset_salvo = entrada
                total_salvo  = None
                offsets[arquivo] = {'offset': offset_salvo, 'total_paginas': total_atual}

            sinal = f'+{offset_salvo}' if offset_salvo >= 0 else str(offset_salvo)

            if total_salvo is not None and total_salvo != total_atual:
                algum_desatualizado = True
                print(f'\n⚠️  OFFSET DESATUALIZADO: {arquivo}')
                print(f'   Salvo com {total_salvo} páginas — arquivo atual tem {total_atual} páginas.')
                print(f'   O PDF pode ter sido substituído. Remapeando...')
                offset_novo, total_novo = _pedir_offset(arquivo, caminho)
                offsets[arquivo] = {'offset': offset_novo, 'total_paginas': total_novo}
                sinal = f'+{offset_novo}' if offset_novo >= 0 else str(offset_novo)
                print(f'   💾 Offset atualizado: {sinal}')
            else:
                offsets[arquivo] = {'offset': offset_salvo, 'total_paginas': total_atual}
                print(f'✅ {arquivo:<40} offset {sinal:>5}  ({total_atual} págs.)')

        else:
            algum_novo = True
            offset, total = _pedir_offset(arquivo, caminho)
            offsets[arquivo] = {'offset': offset, 'total_paginas': total}

    # Salva tudo (incluindo módulos antigos) — mas só exibe os atuais
    _salvar_offsets(offsets)

    print()
    print('=' * 65)
    print('📋 OFFSETS MAPEADOS (cole no prompt do Gemini/Claude):')
    print('=' * 65)
    for arquivo in livros:  # ← só os da pasta atual, não todo o offsets.json
        entrada = offsets[arquivo]
        if isinstance(entrada, dict):
            offset = entrada['offset']
            total  = entrada['total_paginas']
        else:
            offset = entrada
            total  = '?'
        sinal = f'+{offset}' if offset >= 0 else str(offset)
        print(f'  {arquivo:<45} offset {sinal:>5}   ({total} págs.)')
    print('=' * 65)

    if algum_desatualizado:
        print('\n⚠️  Um ou mais offsets foram remapeados porque o arquivo mudou.')
    if not algum_novo and not algum_desatualizado:
        print('\n✅ Todos os offsets já estavam salvos e atualizados.')
    print(f'💾 offsets.json salvo em: {OFFSETS_PATH}')


mapear_offsets()

📐 CÉLULA 7 — OFFSETS DE PÁGINA

📖 Mapeando: MS - Doenças relacionadas ao trabalho.pdf  (580 páginas no arquivo)
   Para calcular o offset, escolha qualquer página com número impresso visível.
   → Número impresso na página: 13
   → Número que o leitor de PDF mostra (contador): 13
   ✅ Offset calculado: +0  (impresso 13 = arquivo 13)
✅ MS - Dor relacionada trabalho.pdf        offset    +1  (70 págs.)


✅ MS - Guia de Vigilância em saúde.pdf   offset    +1  (1128 págs.)
✅ MS - Protocolo Perda Auditiva.pdf        offset    +0  (40 págs.)

📖 Mapeando: MS - Protocolo_pneumoconioses.pdf  (76 páginas no arquivo)
   Para calcular o offset, escolha qualquer página com número impresso visível.
   → Número impresso na página: 7
   → Número que o leitor de PDF mostra (contador): 7
   ✅ Offset calculado: +0  (impresso 7 = arquivo 7)

📖 Mapeando: MS - Saúde do Trabalhador.pdf  (138 páginas no arquivo)
   Para calcular o offset, escolha qualquer página com número impresso visível.
   → Número impresso na página: 13
   → Número que o leitor de PDF mostra (contador): 14
   ✅ Offset calculado: +1  (impresso 13 = arquivo 14)

📋 OFFSETS MAPEADOS (cole no prompt do Gemini/Claude):
  MS - Doenças relacionadas ao trabalho.pdf    offset    +0   (580 págs.)
  MS - Dor relacionada trabalho.pdf             offset    +1   (70 págs.)
  MS - Guia de Vigilância em saúde.pdf        offset    +1   (1128 págs.

## 📋 Célula 8 — Listar arquivos gerados

In [ ]:
arquivos = sorted([f for f in os.listdir(PASTA_SAIDA) if f.endswith('.pdf')])
if arquivos:
    print(f'📂 {len(arquivos)} arquivo(s) em saida/:\n')
    for nome in arquivos:
        kb = os.path.getsize(os.path.join(PASTA_SAIDA, nome)) / 1024
        print(f'   📄 {nome}  ({kb:.0f} KB)')
else:
    print('📭 Nenhum PDF gerado ainda.')